In [73]:
import pandas as pd

# Load cleaned data with parse_dates to ensure datetime64[ns] data type
fact_book = pd.read_csv(r'C:\projects_monish\Hotel Analytics\Hotel-analytics\data\clean\fact_bookings_clean.csv', parse_dates=['booking_date', 'check_in_date', 'checkout_date'])
fact_abook = pd.read_csv(r'C:\projects_monish\Hotel Analytics\Hotel-analytics\data\clean\fact_aggregated_bookings_clean.csv', parse_dates=['check_in_date'])
dim_date = pd.read_csv(r'C:\projects_monish\Hotel Analytics\Hotel-analytics\data\clean\dim_date_clean.csv', parse_dates=['date'])
dim_hotels = pd.read_csv(r'C:\projects_monish\Hotel Analytics\Hotel-analytics\data\clean\dim_hotels_clean.csv')
dim_rooms = pd.read_csv(r'C:\projects_monish\Hotel Analytics\Hotel-analytics\data\clean\dim_rooms_clean.csv')

print("Data loaded successfully with proper data types!")

Data loaded successfully with proper data types!


In [74]:
pip install psycopg2-binary sqlalchemy

Note: you may need to restart the kernel to use updated packages.


In [75]:
from sqlalchemy import create_engine, text

# Step 1: Connect to PostgreSQL
username = "postgres"
password = "tiger"
host = "localhost"
port = "5432"
database = "hotel_analytics"

# Create connection string
engine = create_engine(f"postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}")

print("Connected to PostgreSQL successfully!")

Connected to PostgreSQL successfully!


In [ ]:
#Load data into PostgreSQL tables

# Load fact tables
fact_book.to_sql('fact_bookings', engine, if_exists='replace', index=False)
print("✓ fact_bookings loaded successfully!")

fact_abook.to_sql('fact_aggregated_bookings', engine, if_exists='replace', index=False)
print("✓ fact_aggregated_bookings loaded successfully!")

# Load dimension tables
dim_hotels.to_sql('dim_hotels', engine, if_exists='replace', index=False)
print("✓ dim_hotels loaded successfully!")

dim_rooms.to_sql('dim_rooms', engine, if_exists='replace', index=False)
print("✓ dim_rooms loaded successfully!")

dim_date.to_sql('dim_date', engine, if_exists='replace', index=False)
print("✓ dim_date loaded successfully!")


✓ fact_bookings loaded successfully!
✓ fact_aggregated_bookings loaded successfully!
✓ dim_hotels loaded successfully!
✓ dim_rooms loaded successfully!
✓ dim_date loaded successfully!


In [ ]:
#Verify row counts in PostgreSQL tables
with engine.connect() as conn:
    tables = ['dim_hotels', 'dim_rooms', 'dim_date', 'fact_bookings', 'fact_aggregated_bookings']
    for table in tables:
        result = conn.execute(text(f"SELECT COUNT(*) FROM {table}"))
        count = result.scalar()
        print(f"{table}: {count} rows")

dim_hotels: 25 rows
dim_rooms: 4 rows
dim_date: 92 rows
fact_bookings: 134573 rows
fact_aggregated_bookings: 9194 rows


In [ ]:
#Final verification - Data completeness check
with engine.connect() as conn:
    queries = {
        'dim_hotels': "SELECT COUNT(*) FROM dim_hotels WHERE property_id IS NOT NULL",
        'dim_rooms': "SELECT COUNT(*) FROM dim_rooms WHERE room_id IS NOT NULL",
        'dim_date': "SELECT COUNT(*) FROM dim_date WHERE date IS NOT NULL",
        'fact_bookings': "SELECT COUNT(*) FROM fact_bookings WHERE booking_id IS NOT NULL",
        'fact_aggregated_bookings': "SELECT COUNT(*) FROM fact_aggregated_bookings"
    }
    
    print("\n=== Final Data Integrity Check ===")
    for table_name, query in queries.items():
        result = conn.execute(text(query))
        count = result.scalar()
        print(f"✓ {table_name}: {count} valid records")
    
    print("\n✓ All data successfully loaded to PostgreSQL!")


=== Final Data Integrity Check ===
✓ dim_hotels: 25 valid records
✓ dim_rooms: 4 valid records
✓ dim_date: 92 valid records
✓ fact_bookings: 134573 valid records
✓ fact_aggregated_bookings: 9194 valid records

✓ All data successfully loaded to PostgreSQL!
